In [1]:
import os
import pandas as pd
import torch
import torch.nn as nn
import timm
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))

torch.backends.cudnn.benchmark = True

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [3]:
class AIGenDetDataset(Dataset):

    def __init__(self, shard_path, transform=None):

        self.shard_path = shard_path
        self.transform = transform

        self.labels = pd.read_csv(
            os.path.join(shard_path, "labels.csv")
        )

        print(f"Loaded {len(self.labels)} images")


    def __len__(self):
        return len(self.labels)


    def __getitem__(self, idx):

        row = self.labels.iloc[idx]

        img_path = os.path.join(
            self.shard_path,
            "images",
            row["image_name"]
        )

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(row["label"]).long()

        return image, label

In [4]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(380),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),
    transforms.ColorJitter(0.1,0.1,0.1,0.05),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize(400),
    transforms.CenterCrop(380),
    transforms.ToTensor(),
])

In [5]:
import os

print(os.path.exists("/home/daniel"))
print(os.path.exists("/home/daniel/datasets"))
print(os.path.exists("/home/daniel/datasets/ai-gen"))

False
False
False


In [6]:
# DATASET_PATH = "/mnt/c/Development/ai-gen-detection/shard_0/shard_0"
DATASET_PATH = "/workspace/datasets/ai-gen/mnt/c/Development/ai-gen-detection/shard_0/shard_0"

dataset = AIGenDetDataset(
    DATASET_PATH
)

Loaded 50000 images


In [7]:
img, label = dataset[0]

print("Image shape:", img.size)
print("Label:", label)

Image shape: (3456, 2736)
Label: tensor(0)


In [8]:
# import time

# start = time.time()

# for i in range(2000):
#     _ = dataset[i]

# print("Time:", time.time() - start)

In [9]:
# Create custom dataset classes for train and val with proper transforms
class TransformedDataset(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
    
    def __getitem__(self, idx):
        image, label = self.subset[idx]
        if self.transform:
            image = self.transform(image)
        return image, label
    
    def __len__(self):
        return len(self.subset)

indices = list(range(len(dataset)))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.1,
    random_state=42,
    stratify=dataset.labels["label"]
)

# Create subsets and wrap with proper transforms
train_subset = torch.utils.data.Subset(dataset, train_idx)
val_subset = torch.utils.data.Subset(dataset, val_idx)

train_dataset = TransformedDataset(train_subset, train_transform)
val_dataset = TransformedDataset(val_subset, val_transform)

In [10]:
BATCH_SIZE = 16  # Optimized for RTX 4060 laptop (8GB VRAM)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,      # Multi-worker data loading for better performance
    pin_memory=True,
    prefetch_factor=2   # Prefetch 2 batches per worker
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE * 2,  # Can use larger batch for validation (no gradients)
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    prefetch_factor=2
)

In [11]:
# %pip install --upgrade filelock

In [12]:
model = timm.create_model(
    "efficientnet_b4",
    pretrained=True,
    num_classes=2
)

model = model.to(device)

# model = torch.compile(model)

In [14]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

# Add learning rate scheduler for better convergence
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=5,
    eta_min=1e-6
)

scaler = torch.cuda.amp.GradScaler()

In [15]:
# Clear GPU cache and check memory before training setup
torch.cuda.empty_cache()

print("=== GPU Memory Status ===")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print(f"Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
print(f"Reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
print(f"Free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0)) / 1024**3:.2f} GB")

=== GPU Memory Status ===
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Total VRAM: 8.00 GB
Allocated: 0.07 GB
Reserved: 0.08 GB
Free: 7.91 GB


In [16]:
def train_epoch(loader):

    model.train()

    running_loss = 0

    for images, labels in tqdm(loader):

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)  # More efficient than zero_grad()

        with torch.amp.autocast('cuda'):

            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    return running_loss / len(loader)

In [17]:
def validate(loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in tqdm(loader):

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast('cuda'):
                outputs = model(images)

            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [18]:
EPOCHS = 5

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_loss = train_epoch(train_loader)

    val_acc = validate(val_loader)

    print("Train Loss:", train_loss)
    print("Validation Accuracy:", val_acc)
    print("Learning Rate:", optimizer.param_groups[0]['lr'])
    print(f"GPU Memory: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    
    # Step the scheduler
    scheduler.step()
    
    # Clear cache after each epoch
    torch.cuda.empty_cache()


Epoch 1/5


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 157/157 [01:17<00:00,  2.03it/s]


Train Loss: 0.27100760726754114
Validation Accuracy: 0.9548
Learning Rate: 0.0003
GPU Memory: 0.28 GB

Epoch 2/5


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 157/157 [01:13<00:00,  2.13it/s]


Train Loss: 0.15416599922160257
Validation Accuracy: 0.9582
Learning Rate: 0.0002714480406590546
GPU Memory: 0.28 GB

Epoch 3/5


...

Train Loss: 0.10642446386583157
Validation Accuracy: 0.9618
Learning Rate: 0.00019669804065905458
GPU Memory: 0.28 GB

Epoch 4/5


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 157/157 [01:12<00:00,  2.17it/s]


Train Loss: 0.07689681921751526
Validation Accuracy: 0.9604
Learning Rate: 0.00010430195934094532
GPU Memory: 0.28 GB

Epoch 5/5


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 157/157 [01:12<00:00,  2.17it/s]

Train Loss: 0.05395976254578125
Validation Accuracy: 0.9666
Learning Rate: 2.955195934094536e-05
GPU Memory: 0.28 GB


In [30]:
import os

MODEL_PATH = "/workspace/ai-gen-detection/training/baseline/efficientnet_b4_baseline.pth"

if os.path.exists(MODEL_PATH):
    print(f"Model file '{MODEL_PATH}' already exists. Loading model...")
    model.load_state_dict(torch.load(MODEL_PATH))
    model.to(device)
    model.eval()
    print("Model loaded successfully from disk.")
else:
    print(f"Model file not found. Saving trained model to '{MODEL_PATH}'...")
    torch.save(model.state_dict(), MODEL_PATH)
    print("Model saved successfully.")

Model file not found. Saving trained model to '/workspace/ai-gen-detection/training/baseline/efficientnet_b4_baseline.pth'...
Model saved successfully.


In [28]:
import os

# Check current working directory
print(f"Current working directory: {os.getcwd()}")

Current working directory: /workspace
